In [ ]:
from langgraph.graph import StateGraph, START, END
from langchain_openai import ChatOpenAI
from dotenv import load_dotenv
from typing import TypedDict, Annotated
from langchain_core.messages import BaseMessage, HumanMessage
from langgraph.graph.message import add_messages
from langgraph.checkpoint.memory import MemorySaver #Memorysaver is a RAM based checkpointer. In production, we use databases to store data which is to be used even after the program is closed

In [ ]:
class ChatState(TypedDict):

    messages: Annotated[list[BaseMessage] , add_messages]

In [39]:
llm = ChatOpenAI(model='gpt-4o-mini')

In [40]:
def chat_node(state: ChatState):
    #user quesry
    messages = state['messages']

    #send to llm
    response = llm.invoke(messages)
    # print(response)
# 
    #store to state
    return {'messages': [response]}

In [41]:
checkpointer = MemorySaver()

graph = StateGraph(ChatState)

graph.add_node('chat_node', chat_node)

graph.add_edge(START, 'chat_node')
graph.add_edge('chat_node', END)

chatbot = graph.compile(checkpointer=checkpointer)

In [42]:
# initial_state = {'messages': [HumanMessage(content="What is the capital of India?")]}

# chatbot.invoke(initial_state)['messages'][-1].content

In [ ]:
thread_id = '1'

while True:

    user_message = input("User: ")
    print(f"User: {user_message}")

    if user_message.lower() in ['exit', 'quit']:
        break
    config = {'configurable': {'thread_id': thread_id}} # chatbot.get_state(config) gives the entire chat history
    response  = chatbot.invoke({'messages': [HumanMessage(content=user_message)]}, config = config)
    print('AI:', response['messages'][-1].content)

#### Persistency refers to the ability to save and restore the state of a workflow over time.

- helps in Faault tolerance: If a workflow fails or is interrupted, persistency allows it to be resumed from the last saved state, rather than starting over from scratch.

- Resume: Persistency allows workflows to be paused and resumed at a later time, which is useful for long-running processes or when resources are limited.